# Step 4 - Model Training and Comparison

Project: AI-Based Strength Prediction and Stacking Sequence Optimization of Aerospace Composite Laminates

This notebook trains tensile-strength regression models only. It does not implement stacking optimization.

## 1. Setup

Upload/open the `CompositeAI` project folder in Google Colab. Run from the project root.

In [ ]:
# Optional in Colab if xgboost is unavailable
# !pip install xgboost

from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
print(PROJECT_ROOT)

## 2. Load dataset and feature specification

In [ ]:
import json
import pandas as pd

dataset_path = PROJECT_ROOT / 'data' / 'training' / 'ml_ready_features.csv'
spec_path = PROJECT_ROOT / 'data' / 'training' / 'feature_specification.json'

data = pd.read_csv(dataset_path)
spec = json.loads(spec_path.read_text())
expected_columns = spec['baseline_features'] + [spec['target']]
assert list(data.columns) == expected_columns, 'Dataset/spec column mismatch'
data.shape, data.head()

## 3. Define X and y

In [ ]:
X = data[spec['baseline_features']]
y = data[spec['target']]

categorical_features = spec['categorical_features']
numerical_features = spec['numerical_features']

print(X.shape, y.shape)
print(categorical_features)
print(numerical_features)

## 4. Train/test split

Preprocessing is fitted only through each pipeline after splitting.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 5. Build preprocessing

Matches `src/preprocessing.py`: median numeric imputation, StandardScaler, most-frequent categorical imputation, OneHotEncoder(handle_unknown="ignore").

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def build_preprocessor():
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', make_encoder()),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numerical_features),
        ('cat', categorical_pipe, categorical_features),
    ], remainder='drop', verbose_feature_names_out=False)

## 6. Train models

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=180, random_state=42, n_jobs=-1, min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'ANN/MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=600, random_state=42, early_stopping=True, validation_fraction=0.1),
}

try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror', random_state=42, n_jobs=-1)
except Exception as exc:
    print('XGBoost unavailable:', exc)

def metrics(y_true, pred):
    return {
        'mae': mean_absolute_error(y_true, pred),
        'rmse': np.sqrt(mean_squared_error(y_true, pred)),
        'r2': r2_score(y_true, pred),
    }

trained = {}
predictions = {}
rows = []
for name, model in models.items():
    pipe = Pipeline([('preprocessing', build_preprocessor()), ('model', clone(model))])
    pipe.fit(X_train, y_train)
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    train_m = metrics(y_train, train_pred)
    test_m = metrics(y_test, test_pred)
    rows.append({
        'model': name,
        'train_mae': train_m['mae'], 'test_mae': test_m['mae'],
        'train_rmse': train_m['rmse'], 'test_rmse': test_m['rmse'],
        'train_r2': train_m['r2'], 'test_r2': test_m['r2'],
    })
    trained[name] = pipe
    predictions[name] = test_pred

comparison = pd.DataFrame(rows).sort_values('test_rmse')
comparison

## 7. Evaluate and compare results

In [ ]:
comparison[['model', 'train_mae', 'test_mae', 'train_rmse', 'test_rmse', 'train_r2', 'test_r2']]

## 8. Actual vs predicted and residual plots

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

best_name = comparison.iloc[0]['model']
best_pred = predictions[best_name]
plot_df = pd.DataFrame({
    'actual': y_test.to_numpy(),
    'predicted': best_pred,
    'residual': y_test.to_numpy() - best_pred,
})

min_value = min(plot_df['actual'].min(), plot_df['predicted'].min())
max_value = max(plot_df['actual'].max(), plot_df['predicted'].max())
fig = px.scatter(plot_df, x='actual', y='predicted', title=f'{best_name}: Actual vs Predicted')
fig.add_trace(go.Scatter(x=[min_value, max_value], y=[min_value, max_value], mode='lines', name='Actual = Predicted', line={'dash': 'dash', 'color': 'red'}))
fig.show()

px.scatter(plot_df, x='predicted', y='residual', title=f'{best_name}: Residual Plot').add_hline(y=0, line_dash='dash', line_color='red').show()
px.histogram(plot_df, x='residual', nbins=40, title=f'{best_name}: Prediction Error Distribution').show()

## 9. Outlier experiment

Primary experiment keeps all data. Secondary experiment removes target outliers from training rows only and evaluates on the same untouched test set.

In [ ]:
q1 = y_train.quantile(0.25)
q3 = y_train.quantile(0.75)
iqr = q3 - q1
keep_mask = (y_train >= q1 - 1.5 * iqr) & (y_train <= q3 + 1.5 * iqr)

filtered_pipe = Pipeline([('preprocessing', build_preprocessor()), ('model', clone(models[best_name]))])
filtered_pipe.fit(X_train.loc[keep_mask], y_train.loc[keep_mask])
filtered_pred = filtered_pipe.predict(X_test)
filtered_metrics = metrics(y_test, filtered_pred)

outlier_experiment = pd.DataFrame([
    {'experiment': 'all_data_primary', 'model': best_name, 'train_rows': len(y_train), 'test_rows': len(y_test), 'removed_train_outliers': 0, **metrics(y_test, best_pred)},
    {'experiment': 'outlier_filtered_training_only', 'model': best_name, 'train_rows': int(keep_mask.sum()), 'test_rows': len(y_test), 'removed_train_outliers': int((~keep_mask).sum()), **filtered_metrics},
])
outlier_experiment

## 10. Select best model and save/export artifacts

In [ ]:
import joblib
from datetime import datetime, timezone

best_name = comparison.iloc[0]['model']
best_model = trained[best_name]
best_row = comparison.iloc[0].to_dict()

(PROJECT_ROOT / 'saved_models').mkdir(exist_ok=True)
(PROJECT_ROOT / 'data' / 'training').mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, PROJECT_ROOT / 'saved_models' / 'best_strength_model.joblib')
comparison.to_csv(PROJECT_ROOT / 'data' / 'training' / 'model_comparison.csv', index=False)
outlier_experiment.to_csv(PROJECT_ROOT / 'data' / 'training' / 'outlier_experiment.csv', index=False)

metadata = {
    'model_name': best_name,
    'training_dataset': 'data/training/ml_ready_features.csv',
    'target': spec['target'],
    'feature_list': spec['baseline_features'],
    'train_test_split': {'test_size': 0.20, 'train_rows': len(X_train), 'test_rows': len(X_test)},
    'random_state': RANDOM_STATE,
    'training_date': datetime.now(timezone.utc).isoformat(),
    'metrics': best_row,
}
(PROJECT_ROOT / 'saved_models' / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=str))
best_name, best_row